# Module 3: Calculus — Single Variable

Calculus is the mathematics of **change**. In AI, we need calculus to understand how changing model parameters affects the loss — this is the foundation of **training neural networks**.

### 🎯 What you'll learn:
- Limits and continuity
- Derivatives and differentiation rules
- Chain rule (the heart of backpropagation!)
- Taylor series approximations
- Integrals and the Fundamental Theorem
- Common functions and their derivatives

### 🤖 Why it matters for AI:
- **Gradient descent** = using derivatives to minimize loss
- **Backpropagation** = chain rule applied recursively
- **Taylor series** = how we approximate complex functions
- **Integration** = computing probabilities, expectations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# SymPy for symbolic math
x = sp.Symbol('x')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-darkgrid')

---
## 1. Limits

The **limit** of $f(x)$ as $x$ approaches $a$:
$$\lim_{x \to a} f(x) = L$$

means $f(x)$ gets arbitrarily close to $L$ as $x$ gets close to $a$.

### Key Limits for AI:
$$\lim_{x \to 0} \frac{\sin x}{x} = 1 \qquad \lim_{n \to \infty} \left(1 + \frac{1}{n}\right)^n = e \qquad \lim_{x \to 0} \frac{e^x - 1}{x} = 1$$

In [ ]:
# Numerical limits
print("=== Numerical Limits ===")
for h in [1, 0.1, 0.01, 0.001, 0.0001]:
    val = np.sin(h) / h
    print(f"sin({h})/{h} = {val:.8f}")

print(f"\nLimit = 1.0")

# Symbolic limits with SymPy
print("\n=== Symbolic Limits ===")
print(f"lim(sin(x)/x, x→0) = {sp.limit(sp.sin(x)/x, x, 0)}")
print(f"lim((1+1/x)^x, x→∞) = {sp.limit((1 + 1/x)**x, x, sp.oo)}")
print(f"lim((e^x - 1)/x, x→0) = {sp.limit((sp.exp(x)-1)/x, x, 0)}")

---
## 2. Derivatives — The Rate of Change

The **derivative** of $f$ at point $a$:
$$f'(a) = \lim_{h \to 0} \frac{f(a+h) - f(a)}{h}$$

### Fundamental Differentiation Rules:

| Rule | Formula |
|------|--------|
| Power Rule | $\frac{d}{dx}[x^n] = nx^{n-1}$ |
| Constant Multiple | $\frac{d}{dx}[cf] = cf'$ |
| Sum Rule | $\frac{d}{dx}[f+g] = f' + g'$ |
| Product Rule | $\frac{d}{dx}[fg] = f'g + fg'$ |
| Quotient Rule | $\frac{d}{dx}[f/g] = \frac{f'g - fg'}{g^2}$ |
| Chain Rule | $\frac{d}{dx}[f(g(x))] = f'(g(x)) \cdot g'(x)$ |

### 🤖 AI Connection:
The derivative tells us: **"If I change the input by a tiny amount, how much does the output change?"**
This is exactly what we need for gradient descent!

In [ ]:
# Numerical derivative
def numerical_derivative(f, x, h=1e-7):
    """Compute derivative using central difference (more accurate)"""
    return (f(x + h) - f(x - h)) / (2 * h)

# Example: f(x) = x^3
f = lambda x: x**3
f_prime_numerical = numerical_derivative(f, 2.0)
f_prime_exact = 3 * 2.0**2  # Power rule: 3x^2

print(f"f(x) = x³")
print(f"f'(2) numerical = {f_prime_numerical:.8f}")
print(f"f'(2) exact     = {f_prime_exact:.8f}")

# Symbolic derivatives with SymPy
print("\n=== Symbolic Derivatives ===")
functions = [x**3, sp.sin(x), sp.exp(x), sp.log(x), 1/x]
for func in functions:
    deriv = sp.diff(func, x)
    print(f"d/dx [{func}] = {deriv}")

# Visualize derivative as slope
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = np.linspace(-2, 3, 200)
y_vals = x_vals**3 - 3*x_vals  # f(x) = x³ - 3x
dy_vals = 3*x_vals**2 - 3       # f'(x) = 3x² - 3

ax.plot(x_vals, y_vals, 'b-', linewidth=2, label='f(x) = x³ - 3x')
ax.plot(x_vals, dy_vals, 'r--', linewidth=2, label="f'(x) = 3x² - 3")

# Mark critical points (where f'(x) = 0)
ax.plot([-1, 1], [2, -2], 'ko', markersize=8)
ax.annotate('Local max (f\'=0)', xy=(-1, 2), xytext=(-2, 4), fontsize=11,
           arrowprops=dict(arrowstyle='->'))
ax.annotate('Local min (f\'=0)', xy=(1, -2), xytext=(2, -4), fontsize=11,
           arrowprops=dict(arrowstyle='->'))

ax.axhline(y=0, color='k', linewidth=0.5)
ax.set_xlabel('x', fontsize=14); ax.set_ylabel('y', fontsize=14)
ax.set_title('Function and Its Derivative', fontsize=16)
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
plt.show()

---
## 3. The Chain Rule — The Heart of Backpropagation

If $y = f(g(x))$, then:
$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx} \quad \text{where } u = g(x)$$

Or equivalently: $[f(g(x))]' = f'(g(x)) \cdot g'(x)$

### 🤖 Why this is THE most important rule in AI:

In a neural network: $\text{Loss} = L(\sigma(Wx + b))$

To compute $\frac{\partial L}{\partial W}$, we apply the chain rule repeatedly:

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial z} \cdot \frac{\partial z}{\partial W}$$

This is **backpropagation** — literally just the chain rule applied backwards through the network!

In [ ]:
# Chain rule examples
print("=== Chain Rule ===")

# Example 1: d/dx [sin(x²)] = cos(x²) · 2x
f = sp.sin(x**2)
print(f"d/dx [sin(x²)] = {sp.diff(f, x)}")

# Example 2: d/dx [e^(3x)] = 3e^(3x)
f = sp.exp(3*x)
print(f"d/dx [e^(3x)] = {sp.diff(f, x)}")

# Example 3: d/dx [log(x² + 1)] = 2x/(x² + 1)
f = sp.log(x**2 + 1)
print(f"d/dx [log(x²+1)] = {sp.diff(f, x)}")

# Backpropagation demo: y = sigmoid(w*x + b)
print("\n=== Backpropagation (Chain Rule in Action) ===")
w, b_sym = sp.symbols('w b')
z = w * x + b_sym           # Linear transform
sigma = 1 / (1 + sp.exp(-z))  # Sigmoid activation
L = (sigma - 1)**2           # MSE loss (target = 1)

dL_dw = sp.diff(L, w)
dL_db = sp.diff(L, b_sym)

print(f"z = wx + b")
print(f"σ = 1/(1+e^(-z))")
print(f"L = (σ - 1)²")
print(f"\n∂L/∂w = {sp.simplify(dL_dw)}")
print(f"∂L/∂b = {sp.simplify(dL_db)}")

---
## 4. Common AI Functions and Their Derivatives

These are the functions you'll encounter constantly in AI:

| Function | Formula | Derivative | Use in AI |
|----------|---------|------------|----------|
| Sigmoid | $\sigma(x) = \frac{1}{1+e^{-x}}$ | $\sigma(x)(1-\sigma(x))$ | Binary classification |
| Tanh | $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$ | $1 - \tanh^2(x)$ | Hidden layers |
| ReLU | $\max(0, x)$ | $1$ if $x > 0$, else $0$ | Most common activation |
| Softmax | $\frac{e^{x_i}}{\sum e^{x_j}}$ | (complex) | Multi-class classification |
| Log | $\ln(x)$ | $1/x$ | Cross-entropy loss |

In [ ]:
# AI activation functions and their derivatives
x_vals = np.linspace(-5, 5, 200)

# Sigmoid
sigmoid = 1 / (1 + np.exp(-x_vals))
sigmoid_deriv = sigmoid * (1 - sigmoid)

# Tanh
tanh = np.tanh(x_vals)
tanh_deriv = 1 - tanh**2

# ReLU
relu = np.maximum(0, x_vals)
relu_deriv = (x_vals > 0).astype(float)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Plot functions
for ax, func, name, color in zip(axes[0], [sigmoid, tanh, relu],
                                   ['Sigmoid', 'Tanh', 'ReLU'],
                                   ['#E74C3C', '#3498DB', '#2ECC71']):
    ax.plot(x_vals, func, color=color, linewidth=2.5)
    ax.set_title(name, fontsize=16, fontweight='bold')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)

# Plot derivatives
for ax, deriv, name, color in zip(axes[1], [sigmoid_deriv, tanh_deriv, relu_deriv],
                                    ['Sigmoid\'', 'Tanh\'', 'ReLU\''],
                                    ['#E74C3C', '#3498DB', '#2ECC71']):
    ax.plot(x_vals, deriv, color=color, linewidth=2.5, linestyle='--')
    ax.set_title(f'{name} (Derivative)', fontsize=14)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)

plt.suptitle('AI Activation Functions & Their Derivatives', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Taylor Series — Approximating Functions

Any smooth function can be approximated by a polynomial:

$$f(x) = \sum_{n=0}^{\infty} \frac{f^{(n)}(a)}{n!}(x-a)^n = f(a) + f'(a)(x-a) + \frac{f''(a)}{2!}(x-a)^2 + \cdots$$

### Important Taylor Series:
$$e^x = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \cdots$$

$$\sin(x) = x - \frac{x^3}{3!} + \frac{x^5}{5!} - \cdots$$

$$\frac{1}{1-x} = 1 + x + x^2 + x^3 + \cdots \quad (|x| < 1)$$

### 🤖 AI Connection:
- **Second-order optimization** uses the 2nd-order Taylor expansion of the loss
- **GELU** activation uses a Taylor approximation
- **Softmax temperature scaling** uses Taylor expansion

In [ ]:
# Taylor series approximation of sin(x)
x_vals = np.linspace(-2*np.pi, 2*np.pi, 300)
y_true = np.sin(x_vals)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(x_vals, y_true, 'k-', linewidth=3, label='sin(x) exact', alpha=0.8)

colors = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6', '#F39C12']
for n, color in zip([1, 3, 5, 7, 9], colors):
    # Taylor series of sin(x) up to order n
    taylor = sum((-1)**k * x_vals**(2*k+1) / np.math.factorial(2*k+1) 
                 for k in range((n+1)//2))
    ax.plot(x_vals, taylor, '--', color=color, linewidth=1.5, label=f'Order {n}')

ax.set_ylim(-3, 3)
ax.set_xlabel('x', fontsize=14); ax.set_ylabel('y', fontsize=14)
ax.set_title('Taylor Series Approximation of sin(x)', fontsize=16)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.show()

# Symbolic Taylor series
x_sym = sp.Symbol('x')
print("Taylor series (SymPy):")
print(f"e^x ≈ {sp.series(sp.exp(x_sym), x_sym, 0, 6)}")
print(f"sin(x) ≈ {sp.series(sp.sin(x_sym), x_sym, 0, 8)}")
print(f"cos(x) ≈ {sp.series(sp.cos(x_sym), x_sym, 0, 8)}")

---
## 6. Integration — The Reverse of Differentiation

### Indefinite Integral
$$\int f(x) \, dx = F(x) + C \quad \text{where } F'(x) = f(x)$$

### Definite Integral (Fundamental Theorem of Calculus)
$$\int_a^b f(x) \, dx = F(b) - F(a)$$

### Key Integrals:
$$\int x^n \, dx = \frac{x^{n+1}}{n+1} + C \qquad \int e^x \, dx = e^x + C \qquad \int \frac{1}{x} \, dx = \ln|x| + C$$

### 🤖 AI Connection:
- **Probability**: $P(a \leq X \leq b) = \int_a^b f(x) \, dx$
- **Expected value**: $E[X] = \int x f(x) \, dx$
- **Evidence** in Bayesian inference: $P(D) = \int P(D|\theta) P(\theta) \, d\theta$

In [ ]:
# Symbolic integration
print("=== Indefinite Integrals ===")
integrals = [x**3, sp.sin(x), sp.exp(x), 1/x, sp.cos(x)]
for func in integrals:
    result = sp.integrate(func, x)
    print(f"∫ {func} dx = {result} + C")

# Definite integral
print("\n=== Definite Integrals ===")
result = sp.integrate(x**2, (x, 0, 3))
print(f"∫₀³ x² dx = {result}")

# Gaussian integral (fundamental in probability!)
result = sp.integrate(sp.exp(-x**2), (x, -sp.oo, sp.oo))
print(f"∫₋∞^∞ e^(-x²) dx = {result} (= √π)")

# Numerical integration
from scipy import integrate

# Probability: P(-1 ≤ X ≤ 1) for standard normal
f = lambda x: (1/np.sqrt(2*np.pi)) * np.exp(-x**2/2)
prob, error = integrate.quad(f, -1, 1)
print(f"\nP(-1 ≤ X ≤ 1) for N(0,1) = {prob:.6f} (≈ 68.27%)")

# Visualize integral as area under curve
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = np.linspace(-4, 4, 300)
y_vals = f(x_vals)
ax.plot(x_vals, y_vals, 'b-', linewidth=2, label='Standard Normal PDF')
x_fill = np.linspace(-1, 1, 100)
ax.fill_between(x_fill, f(x_fill), alpha=0.3, color='#E74C3C', label=f'P(-1≤X≤1) = {prob:.4f}')
ax.set_xlabel('x', fontsize=14); ax.set_ylabel('f(x)', fontsize=14)
ax.set_title('Integration = Area Under the Curve', fontsize=16)
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
plt.show()

---
## 7. L'Hôpital's Rule

When a limit gives $\frac{0}{0}$ or $\frac{\infty}{\infty}$:

$$\lim_{x \to a} \frac{f(x)}{g(x)} = \lim_{x \to a} \frac{f'(x)}{g'(x)}$$

(provided the right-hand limit exists)

In [ ]:
# L'Hôpital's Rule examples
x_sym = sp.Symbol('x')

# sin(x)/x as x→0 (0/0 form)
print(f"lim sin(x)/x = {sp.limit(sp.sin(x_sym)/x_sym, x_sym, 0)}")

# (e^x - 1)/x as x→0 (0/0 form)
print(f"lim (e^x - 1)/x = {sp.limit((sp.exp(x_sym)-1)/x_sym, x_sym, 0)}")

# x*ln(x) as x→0+ (0·(-∞) form, rewrite as ln(x)/(1/x))
print(f"lim x·ln(x) as x→0+ = {sp.limit(x_sym*sp.log(x_sym), x_sym, 0, '+')}")

# x·e^(-x) as x→∞ (∞·0 form)
print(f"lim x·e^(-x) as x→∞ = {sp.limit(x_sym*sp.exp(-x_sym), x_sym, sp.oo)}")

---
## 8. Gradient Descent — Calculus in Action! 🚀

**Gradient descent** uses derivatives to find the minimum of a function:

$$x_{t+1} = x_t - \alpha \cdot f'(x_t)$$

Where $\alpha$ is the **learning rate**.

### Intuition:
- If $f'(x) > 0$: function is increasing → move **left** (decrease $x$)
- If $f'(x) < 0$: function is decreasing → move **right** (increase $x$)
- If $f'(x) = 0$: at a critical point (possibly minimum!)

In [ ]:
# Gradient descent on f(x) = (x - 3)² + 1
def f(x):
    return (x - 3)**2 + 1

def f_prime(x):
    return 2 * (x - 3)

# Run gradient descent
x = -2.0  # Starting point
lr = 0.1  # Learning rate
history = [x]

for step in range(30):
    grad = f_prime(x)
    x = x - lr * grad
    history.append(x)

history = np.array(history)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot function with gradient descent path
x_vals = np.linspace(-3, 7, 200)
ax1.plot(x_vals, f(x_vals), 'b-', linewidth=2, label='f(x) = (x-3)² + 1')
ax1.plot(history, f(history), 'ro-', markersize=4, alpha=0.7, label='GD path')
ax1.plot(history[0], f(history[0]), 'g*', markersize=15, label='Start')
ax1.plot(history[-1], f(history[-1]), 'r*', markersize=15, label=f'End (x={history[-1]:.4f})')
ax1.set_xlabel('x', fontsize=14); ax1.set_ylabel('f(x)', fontsize=14)
ax1.set_title('Gradient Descent Finding the Minimum', fontsize=14)
ax1.legend(fontsize=11); ax1.grid(True, alpha=0.3)

# Plot convergence
ax2.plot(f(history), 'r-o', markersize=3)
ax2.set_xlabel('Step', fontsize=14); ax2.set_ylabel('f(x)', fontsize=14)
ax2.set_title('Loss Convergence', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nStarted at x = -2.0, f(-2) = {f(-2):.2f}")
print(f"Converged to x = {history[-1]:.6f}, f(x) = {f(history[-1]):.6f}")
print(f"True minimum at x = 3.0, f(3) = 1.0")

---
## 9. Higher-Order Derivatives

The **second derivative** $f''(x)$ tells us about **curvature**:
- $f''(x) > 0$: concave up (U-shape) → local **minimum**
- $f''(x) < 0$: concave down (∩-shape) → local **maximum**
- $f''(x) = 0$: inflection point

### Second Derivative Test:
At a critical point $x_0$ where $f'(x_0) = 0$:
- $f''(x_0) > 0$ → local minimum
- $f''(x_0) < 0$ → local maximum

### 🤖 AI Connection:
The second derivative becomes the **Hessian matrix** in multiple dimensions (Module 4), which Newton's method uses for faster optimization.

In [ ]:
# Higher-order derivatives
x_sym = sp.Symbol('x')
f = x_sym**4 - 4*x_sym**3 + 4*x_sym**2

f1 = sp.diff(f, x_sym)     # First derivative
f2 = sp.diff(f, x_sym, 2)  # Second derivative
f3 = sp.diff(f, x_sym, 3)  # Third derivative

print(f"f(x) = {f}")
print(f"f'(x) = {f1}")
print(f"f''(x) = {f2}")
print(f"f'''(x) = {f3}")

# Find critical points
critical = sp.solve(f1, x_sym)
print(f"\nCritical points: {critical}")
for cp in critical:
    second_deriv = f2.subs(x_sym, cp)
    nature = 'minimum' if second_deriv > 0 else ('maximum' if second_deriv < 0 else 'inflection')
    print(f"  x={cp}: f''={second_deriv} → {nature}")

---
## 10. Summary: Calculus for AI

| Concept | AI Application |
|---------|---------------|
| Derivative | Gradient computation |
| Chain Rule | Backpropagation |
| Taylor Series | Function approximation, GELU |
| Integration | Probability, expectations |
| Second Derivative | Newton's method, loss curvature |
| Gradient Descent | Training every ML model |

**Next: Multivariable calculus, where we handle functions of many variables (as neural networks have millions of parameters!)** 🚀